In [1]:
# Import standard libraries
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment
from envs.balance_bot_env import BalanceBotEnv

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 1   # Single environment for now (so we can watch it train)

In [4]:
# Create env and warm up MuJoCo's LLVM shader compilation
render_env = RecordEpisodeStatistics(
    BalanceBotEnv(mjcf_path=MJCF_PATH, render_mode="human")
)
envs = gym.vector.SyncVectorEnv([lambda: render_env])
envs.envs[0].render()  # warm up LLVM before TensorFlow loads
print("MuJoCo ready")

MuJoCo ready


In [5]:
envs.close()

In [6]:
# Cell 2 — ppo_trainer import (TensorFlow/LLVM loads here)
from rl.ppo_trainer import train, PPOConfig
print("ppo_trainer ready")

I0000 00:00:1777662082.177083    6863 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777662082.207315    6863 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777662083.692039    6863 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


ppo_trainer ready


In [7]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = 3_500,     # Total simulation steps across all envs and iterations
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.002,              # Match MJCF opt.timestep for real-time rendering
)

In [8]:
# Build the environment
env = BalanceBotEnv(
    mjcf_path    = MJCF_PATH,
    render_mode  = "human",
)

# Wrap in RecordEpisodeStatistics so we can log episodic returns
env = gym.wrappers.RecordEpisodeStatistics(env)

# Wrap in SyncVectorEnv so it's compatible with train()
envs = gym.vector.SyncVectorEnv([lambda: env])

In [9]:
# TEST: Does render work on the same envs object we're about to pass to train()?
print(f"envs type: {type(envs)}")
print(f"envs.envs[0] type: {type(envs.envs[0])}")
print(f"render_mode: {envs.envs[0].render_mode}")
print("attempting render...")
envs.envs[0].render()
print("render OK!")

envs type: <class 'gymnasium.vector.sync_vector_env.SyncVectorEnv'>
envs.envs[0] type: <class 'gymnasium.wrappers.common.RecordEpisodeStatistics'>
render_mode: human
attempting render...
render OK!


In [10]:
envs.close()

In [11]:
# Choo choo train!
agent = train(ppo_config, envs=envs)

START: train()
CHECKPOINT: config computed
CHECKPOINT: writer initialized
CHECKPOINT: envs ready
CHECKPOINT: agent created
CHECKPOINT: storage allocated
CHECKPOINT: initial reset done
CHECKPOINT: attempting test render
  envs.envs[0] type: <class 'gymnasium.wrappers.common.RecordEpisodeStatistics'>
  render_mode: human
CHECKPOINT: test render complete
CHECKPOINT: entering iteration loop
CHECKPOINT: iteration 1 start
CHECKPOINT: step 0
CHECKPOINT: step 1
CHECKPOINT: step 2
CHECKPOINT: step 3
CHECKPOINT: step 4
CHECKPOINT: step 5
CHECKPOINT: step 6
CHECKPOINT: step 7
CHECKPOINT: step 8
CHECKPOINT: step 9
CHECKPOINT: step 10
CHECKPOINT: step 11
CHECKPOINT: step 12
CHECKPOINT: step 13
CHECKPOINT: step 14
CHECKPOINT: step 15
CHECKPOINT: step 16
CHECKPOINT: step 17
CHECKPOINT: step 18
CHECKPOINT: step 19
CHECKPOINT: step 20
CHECKPOINT: step 21
CHECKPOINT: step 22
CHECKPOINT: step 23
CHECKPOINT: step 24
CHECKPOINT: step 25
CHECKPOINT: step 26
CHECKPOINT: step 27
CHECKPOINT: step 28
CHECKPOINT

In [1]:
# Cell 1: does wrapping in SyncVectorEnv crash?
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
from balance_bot_env import BalanceBotEnv
from pathlib import Path

MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")

render_env = RecordEpisodeStatistics(BalanceBotEnv(mjcf_path=MJCF_PATH, render_mode="human"))
envs = gym.vector.SyncVectorEnv([lambda: render_env])
print("SyncVectorEnv created OK")

Filesystem      Size  Used Avail Use% Mounted on
shm             2.0G     0  2.0G   0% /dev/shm



In [13]:
# Cell 2: does reset crash?
obs, _ = envs.reset(seed=42)
print(f"reset OK, obs shape: {obs.shape}")

reset OK, obs shape: (1, 4)


In [14]:
# Cell 3: does a single step crash?
import numpy as np
action = envs.action_space.sample()
obs, reward, terminated, truncated, infos = envs.step(action)
print(f"step OK, reward: {reward}")

step OK, reward: [0.99341118]


In [15]:
# Cell 4: does importing ppo_trainer crash?
import sys
sys.path.append("/workspace/software")
from rl.ppo_trainer import train, PPOConfig
print("import OK")

import OK
